In [ ]:
!pip install speechbrain

In [ ]:
!pip install torchmetrics[audio]

In [ ]:
!pip install https://github.com/ludlows/python-pesq/archive/master.zip

In [ ]:
import pandas as pd
import librosa

from torchmetrics.audio import SignalDistortionRatio
from speechbrain.inference.separation import SepformerSeparation as separator
import numpy as np

In [ ]:
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality

In [ ]:
SAMPLE_RATE = 16000

In [ ]:
def get_permutations():
    perms = [
        (sdr(x1, pred1) + sdr(x2, pred2)) / 2,
        (sdr(x1, pred2) + sdr(x2, pred1)) / 2
    ]
    best_score = max(perms)

sdr = SignalDistortionRatio()
def calculate_sdr(preds, target):
    return sdr(preds, target)

wb_pesq = PerceptualEvaluationSpeechQuality(16000, 'wb')
def calculate_wb_pesq(preds, target):
    return wb_pesq(preds, target)

def read_file(file_path: str):
    waveform, sample_rate = librosa.load(file_path)
    if sample_rate != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=SAMPLE_RATE)
    return waveform
    

In [ ]:
model = separator.from_hparams(source="speechbrain/resepformer-wsj02mix", savedir='pretrained_models/resepformer-wsj02mix')

In [ ]:
from pathlib import Path
from tqdm import tqdm
import time

def process_dataset(df, model, model_name, model_sample_rate=8000, out_dir='output'):
    out1_dir = Path(out_dir + "_" + model_name) / "out1"
    out2_dir = Path(out_dir + "_" + model_name) / "out2"
    out1_dir.mkdir(parents=True, exist_ok=True)
    out2_dir.mkdir(parents=True, exist_ok=True)
    separation_times = []

    for idx, row in tqdm(df.iterrows()):
        mix_path = row['ID']
        filename = Path(mix_path).stem

        input_wave = read_file(filename)
        if mix_wave.dim() > 1:  # первод в 1 канал
            mix_wave = mix_wave.mean(dim=0, keepdim=True)

        st = time.time()
        est_sources = model.separate(mix_wave, sr)  # [2, T] — размерность выхода
        separation_times.append(time.time() - st)

        torchaudio.save(str(out1_dir / f"{filename}_1.wav"), est_sources[:, :, 0].detach().cpu(), model_sample_rate)
        torchaudio.save(str(out2_dir / f"{filename}_2.wav"), est_sources[:, :, 1].detach().cpu(), model_sample_rate)
    return separation_times

In [ ]:
def compute_file_metrics(row: dict, refs: np.ndarray, hyps: np.ndarray, sr: int) -> tuple:
    s0, e0 = int(row['start1']), int(row['end1'])
    s1, e1 = int(row['start2']), int(row['end2'])

    ref0 = refs[0, s0:e0]
    ref1 = refs[1, s1:e1]

    best_sdr = -np.inf
    best_pesq = -np.inf

    for perm in [(0, 1), (1, 0)]:
        hy0 = hyps[perm[0]]
        hy1 = hyps[perm[1]]
        hyp0 = hy0[s0:e0]
        hyp1 = hy1[s1:e1]

        vals = []
        pesqs = []
        for ref_sig, hyp_sig in [(ref0, hyp0), (ref1, hyp1)]:
            if len(ref_sig) < sr // 10:
                continue
            vals.append(single_channel_sdr(ref_sig, hyp_sig))
            pesqs.append(pesq(sr, ref_sig, hyp_sig, 'wb'))

        if not vals:
            continue

        avg_sdr = float(np.mean(vals))
        avg_pesq = float(np.mean(pesqs))

        if avg_sdr > best_sdr:
            best_sdr = avg_sdr
            best_pesq = avg_pesq

    if best_sdr == -np.inf:
        return 0.0, 0.0
    return best_sdr, best_pesq

In [ ]:
def evaluate_model(df, model, model_name, model_sample_rate: int = 8000, out_dir: str = 'output') -> tuple:
    sdr_scores = []
    pesq_scores = []
    base_dir = Path(f"{out_dir}_{model_name}")
    out1_dir = base_dir / "out1"
    out2_dir = base_dir / "out2"

    for _, row in tqdm(df.iterrows(), total=len(df)):
        hyp1, sr_h1 = torchaudio.load(out1_dir / row['file1'])
        hyp2, sr_h2 = torchaudio.load(out2_dir / row['file2'])
        hyps = np.stack([hyp1.mean(dim=0).numpy(), hyp2.mean(dim=0).numpy()])

        ref1, sr_r1 = torchaudio.load(row['path1'])
        ref2, sr_r2 = torchaudio.load(row['path2'])
        refs = np.stack([ref1.mean(dim=0).numpy(), ref2.mean(dim=0).numpy()])

        sdr_val, pesq_val = compute_file_metrics(row, refs, hyps, sr)
        sdr_scores.append(sdr_val)
        pesq_scores.append(pesq_val)

    return sdr_scores, pesq_scores